In [28]:
!pwd

2309.11s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
/kaggle/working


In [29]:
!nvidia-smi

2317.98s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
Wed Sep  9 12:56:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             29W /   70W |   11685MiB /  15360MiB |      0%      Default |
|                                         |  

In [30]:
!ls

2328.22s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
vllm.log


In [17]:
!ps aux | grep -E "vllm|api_server" | grep -v grep

root         225 10.6  6.1 8780100 2018280 ?     Sl   12:25   0:48 /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-VL-2B-Instruct --served-model-name arc-local --host 127.0.0.1 --port 1234 --dtype half --max-model-len 8192 --gpu-memory-utilization 0.90


In [18]:
import requests

r = requests.get(
    "http://127.0.0.1:1234/v1/models",
    timeout=10,
)

print(r.status_code)
print(r.json())

200
{'object': 'list', 'data': [{'id': 'arc-local', 'object': 'model', 'created': 1788957192, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3-VL-2B-Instruct', 'parent': None, 'max_model_len': 8192, 'permission': [{'id': 'modelperm-a938da1ad348222f', 'object': 'model_permission', 'created': 1788957192, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [19]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="local",
)

response = client.chat.completions.create(
    model="arc-local",
    messages=[
        {
            "role": "user",
            "content": "Reply exactly with: LOCAL MODEL WORKS",
        }
    ],
    temperature=0,
    max_tokens=50,
)

print(response.choices[0].message.content)

LOCAL MODEL WORKS


In [21]:
from PIL import Image, ImageDraw
import io
import base64

image = Image.new(
    "RGB",
    (256, 256),
    "white",
)

draw = ImageDraw.Draw(image)

draw.rectangle(
    (50, 50, 200, 200),
    fill="red",
)

buffer = io.BytesIO()
image.save(buffer, format="PNG")

encoded = base64.b64encode(
    buffer.getvalue()
).decode("ascii")

image_url = (
    "data:image/png;base64,"
    + encoded
)

response = client.chat.completions.create(
    model="arc-local",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What shape and color do you see?",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                    },
                },
            ],
        }
    ],
    temperature=0,
    max_tokens=100,
)

print(response.choices[0].message.content)

Based on the image provided, I can see the following:

-   **Shape:** The object is a square.
-   **Color:** The color is red.

The image is a simple, solid red square.


In [31]:
import os

os.environ["ARC_LOCAL_BASE_URL"] = (
    "http://127.0.0.1:1234/v1"
)

os.environ["ARC_LOCAL_MODEL"] = (
    "arc-local"
)

In [2]:
%cd /kaggle/working
!git clone https://github.com/14yamahi/ARC-AGI-3_dev.git

/kaggle/working
Cloning into 'ARC-AGI-3_dev'...
remote: Enumerating objects: 734, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 734 (delta 0), reused 0 (delta 0), pack-reused 728 (from 2)
Receiving objects: 100% (734/734), 489.41 KiB | 13.59 MiB/s, done.
Resolving deltas: 100% (490/490), done.


In [5]:
%cd /kaggle/working/ARC-AGI-3_dev

!ls

/kaggle/working/ARC-AGI-3_dev
agents	 llms.txt  pyproject.toml  README.md  uv.lock
LICENSE  main.py   pytest.ini	   tests


In [6]:
import os

os.environ["ARC_LOCAL_BASE_URL"] = (
    "http://127.0.0.1:1234/v1"
)
os.environ["ARC_LOCAL_MODEL"] = "arc-local"

In [16]:
!git pull
!MPLBACKEND=Agg UV_LINK_MODE=copy uv run main.py --agent=myagent2 --game=ls20

Already up to date.
Traceback (most recent call last):
  File "/kaggle/working/ARC-AGI-3_dev/main.py", line 20, in <module>
    from agents import AVAILABLE_AGENTS, Swarm
  File "/kaggle/working/ARC-AGI-3_dev/agents/__init__.py", line 31, in <module>
    AVAILABLE_AGENTS["reasoningagent"] = ReasoningAgent
                                         ^^^^^^^^^^^^^^
NameError: name 'ReasoningAgent' is not defined


In [14]:
!sed -n '1,240p' agents/__init__.py

from typing import Type, cast

from dotenv import load_dotenv

from .agent import Agent, Playback
from .recorder import Recorder
from .swarm import Swarm
from .templates.langgraph_functional_agent import LangGraphFunc, LangGraphTextOnly
from .templates.langgraph_random_agent import LangGraphRandom
from .templates.langgraph_thinking import LangGraphThinking
from .templates.llm_agents import LLM, FastLLM, GuidedLLM, ReasoningLLM
from .templates.multimodal import MultiModalLLM
from .templates.openclaw_agent import OpenClaw
from .templates.random_agent import Random
from .templates.reasoning_agent import ReasoningAgent
from .templates.smolagents import SmolCodingAgent, SmolVisionAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    cls.__name__.lower(): cast(Type[Agent], cls)
    for cls in Agent.__subclasses__()
    if cls.__name__ != "Playback"
}

# add all the recording files as valid agent names
for rec in Recorder.list():
    AVAILABLE_AGENTS[rec] = Playback

# update